In [1]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.abspath(".."))

In [2]:
from baghchal import BaghChalGame
from training.mcts import MCTS
from training.mcts import MCTSNode
from models.neural_network import create_bagh_chal_model
import numpy as np

In [3]:
# Initialize BaghChalGame
game = BaghChalGame()
print("Initial board:")
game.display_board()

Initial board:
T . . . T
. . . . .
. . . . .
. . . . .
T . . . T



In [4]:
# Test BaghChalGame
assert game.goats_placed == 0, "Initial goats placed should be 0."
assert game.captured_goats == 0, "Initial captured goats should be 0."
assert game.check_victory_conditions() == "ongoing", "Game should start as 'ongoing'."

Generating valid moves for piece_type: 1
Goats placed: 0, Phase: Goat Placement
Not in goat placement phase or invalid piece type.
Found piece at: (0, 0)
Valid move: ((0, 0), (0, 1))
Valid move: ((0, 0), (1, 1))
Found piece at: (0, 4)
Valid move: ((0, 4), (0, 3))
Valid move: ((0, 4), (1, 3))
Found piece at: (4, 0)
Valid move: ((4, 0), (3, 0))
Valid move: ((4, 0), (4, 1))
Valid move: ((4, 0), (3, 1))
Found piece at: (4, 4)
Valid move: ((4, 4), (3, 4))
Valid move: ((4, 4), (4, 3))
Valid move: ((4, 4), (3, 3))


In [5]:
# Initialize and test neural network
model = create_bagh_chal_model(action_space_size=50)
dummy_input = np.random.random((1, 3, 5, 5))
policy, value = model.predict(dummy_input)
print("Policy output shape:", policy.shape)
print("Value output shape:", value.shape)

assert policy.shape == (1, 50), "Policy output shape mismatch."
assert value.shape == (1, 1), "Value output shape mismatch."

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
Policy output shape: (1, 50)
Value output shape: (1, 1)


In [6]:
# Serialize and deserialize game state to test integrity
original_board = game.board.copy()
serialized_state = game.serialize_state_binary()
game.deserialize_state_binary(serialized_state)
print("Deserialized board:")
game.display_board()

Deserialized board:
[[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 1]]
Deserialized board state (type: <class 'list'>): [[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 1]]
Deserialized goats_placed: 0, captured_goats: 0, status: ongoing
Deserialized board:
T . . . T
. . . . .
. . . . .
. . . . .
T . . . T



In [7]:
assert original_board == game.board, "The board state should remain the same after serialization and deserialization."
assert game.status == "ongoing", "The game status should be 'ongoing' after deserialization."

In [8]:
# Test MCTS expansion
mcts = MCTS(game, model)
root_state = game.serialize_state_binary()
root_node = MCTSNode(state=root_state)

# Debugging output for valid moves and expansion
print(f"Expanding node with state: {root_state}")
mcts._expand(root_node)  # Test the _expand method directly
print(f"Children after expansion: {root_node.children}")

Expanding node with state: (282574488338689, 0, 0, 0)
Expanding node with state: (282574488338689, 0, 0, 0)
Deserialized board:
[[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 1]]
Deserialized board state (type: <class 'list'>): [[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 1]]
Deserialized goats_placed: 0, captured_goats: 0, status: ongoing
Game board after deserialization:
[[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 1]]
Goats placed: 0, Phase: Goat Placement
Goats placed: 0, Captured goats: 0, Phase: Goat Placement
Goats placed: 0, Phase: Goat Placement
Current player: goat (placement phase)
Current player: goat
Piece type used for valid moves: 2
Generating valid moves for piece_type: 2
Goats placed: 0, Phase: Goat Placement
Entering goat placement phase
Checking cell (0, 0): 1 (type: <class 'int'>)
Cell (0, 0) is occupied: 1
Checking cell (0, 1): 0 (type: <class 'int'>)
Valid 

In [9]:
assert len(root_node.children) > 0, "Children should be added during expansion."

In [10]:
# Test full MCTS search
print("Running MCTS search...")
mcts_root = mcts.search(root_state, simulations=10)
print(f"Root node children after MCTS search: {len(mcts_root.children)}")

assert len(mcts_root.children) > 0, "MCTS should expand at least one node during search."

Running MCTS search...
Root state: (282574488338689, 0, 0, 0)
Node fully expanded: False
Deserialized board:
[[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 1]]
Deserialized board state (type: <class 'list'>): [[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 1]]
Deserialized goats_placed: 0, captured_goats: 0, status: ongoing
Current game state: ongoing
Expanding node with state: (282574488338689, 0, 0, 0)
Deserialized board:
[[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 1]]
Deserialized board state (type: <class 'list'>): [[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 1]]
Deserialized goats_placed: 0, captured_goats: 0, status: ongoing
Game board after deserialization:
[[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 1]]
Goats placed: 0, Phase: Goat Placement
Goats placed: 0, Captured goats: 0, Phase: Goat Placement
